<a href="https://colab.research.google.com/github/ShiX27/ds2002-fa26/blob/main/notebooks/04-json-apis/2026-10-02%20%E2%80%94%20Normalize%20and%20Join%20Nested%20JSON%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [1]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-08','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is:** one line item in an order.

**Code Explanation:** I changed the `vendor_id` for order 3 from `V-01` to `V-08` to create an unmatched vendor for the join.

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [2]:
import pandas as pd
# TODO: pd.json_normalize(orders_json, record_path='lines', meta=[...])
lines = pd.json_normalize(orders_json, record_path='lines', meta=['order', 'vendor_id'])

# TODO: assert the row count and the quantity total
assert len(lines) == 4
assert lines['qty'].sum() == 7
print('checks passed.')
lines


checks passed.


,item,qty,order,vendor_id
0,Cheeseburger,2,1,V-01
1,Soda,1,1,V-01
2,Foam Finger,1,2,V-10
3,Hot Dog,3,3,V-08


**Code Explanation:** First, convert the nested JSON data in `orders_json` into a flat DataFrame using `pd.json_normalize()` and store the result in `lines`. Then, use `record_path='lines'` to make each line item a row and `meta=['order', 'vendor_id']` to keep the order number and vendor ID in each row. Second, assert that the row count is 4 and the total quantity is 7. If both conditions are correct, print `'checks passed.'` Finally, display the `lines` table.

### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [3]:
joined = pd.merge(lines, pd.json_normalize(vendors_json), on='vendor_id', how='left', indicator=True)
joined

,item,qty,order,vendor_id,name,zone,_merge
0,Cheeseburger,2,1,V-01,Hoos Burgers,A,both
1,Soda,1,1,V-01,Hoos Burgers,A,both
2,Foam Finger,1,2,V-10,Cav Merch,A,both
3,Hot Dog,3,3,V-08,NaN,NaN,left_only


Vendor V-08 did not match, and 3 units are attached to it.

**Code Explanation:** First, convert the nested JSON data in `vendors_json` into a flat DataFrame using `pd.json_normalize()` and left-merge it with the `lines` table on `vendor_id`. Use `indicator=True` to add a `_merge` column that shows whether each row matched in both tables or came only from the `lines` table, and store the merged result in `joined`. Then, display the `joined` table.

### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [4]:
# Label the unmatched vendor as "Unknown Vendor" so its quantity is still included in the table.
joined["name"] = joined["name"].fillna("Unknown Vendor")

vendor_qty = joined.groupby("name")["qty"].sum().reset_index()
vendor_qty

,name,qty
0,Cav Merch,1
1,Hoos Burgers,3
2,Unknown Vendor,3


Cav Merch has a total quantity of 1, Hoos Burgers has a total quantity of 3, and Unknown Vendor has a total quantity of 3.

I kept the unmatched vendor V-08 and labeled its `name` as `"Unknown Vendor"` rather than removing it, so its quantity is still included in the table. Removing it would understate the total quantity. Using a clear label instead of leaving the value as `NaN` also makes the unmatched vendor easier to identify and review.

**Code Explanation:** First, label the `name` of the unmatched vendor as `"Unknown Vendor"` so its quantity is still included in the `joined` table. Second, group the `joined` table by `name`, calculate the total `qty` for each vendor using `.sum()`, and use `.reset_index()` to change `name` from the index back into a regular column. Store the result in `vendor_qty`. Finally, display the `vendor_qty` table.

### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [5]:
zone_qty = joined.groupby("zone")["qty"].sum().reset_index()
zone_qty

,zone,qty
0,A,4


Zone A sold the most items, with 4 units. The zone totals add up to 4 instead of 7 because the unmatched vendor V-08 has 3 units but no zone, so those 3 units are not included in the zone total.

**Code Explanation:** Group the `joined` table by `zone`, calculate the total `qty` for each zone using `.sum()`, and use `.reset_index()` to change `zone` from the index back into a regular column. Store the result in `zone_qty`. Then, display the `zone_qty` table.

### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [6]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]
for order in ragged_json:
    if 'lines' not in order:
        order['lines'] = []
ragged_lines = pd.json_normalize(ragged_json, record_path='lines', meta=['order', 'vendor_id'])
ragged_lines["qty_given"] = ragged_lines["qty"].notna()
ragged_lines

,item,qty,order,vendor_id,qty_given
0,Soda,2.0,4,V-01,True
1,Fries,NaN,6,V-01,False


Order 5 dropped out because it has no line items. The Fries row remains in the table with `qty = NaN`, indicating that the quantity was not provided. This keeps a missing quantity distinguishable from a quantity of 0.

**Code Explanation:** Given the JSON data in `ragged_json`, first create a `for` loop to check each order and add an empty `lines` list to any order that does not have the `lines` key (order 5 in this case), so `pd.json_normalize()` can run without crashing. Then, convert the nested JSON data in `ragged_json` into a flat DataFrame using `pd.json_normalize()` and store the result in `ragged_lines`. Use `record_path='lines'` to make each line item a row and `meta=['order', 'vendor_id']` to keep the order number and vendor ID in each row. For the line item that is missing its `qty` value (`Fries`), keep `qty` as `NaN`. Next, create a `qty_given` column using `.notna()` to show whether a quantity was provided for each line item (`True` or `False`). Finally, display the `ragged_lines` table.

### Q6 — Validate

In [7]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


**Code Explanation:** Uncomment the `assert` statements and check that the row count of the `lines` table is 4, the total quantity in the `lines` table is 7, and the `joined` table has the same number of rows as the `lines` table. If all conditions are correct, print `'checks passed.'`

### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

a) Having one row per line item makes it easier to work with each item separately. It also makes questions like "What is the total quantity sold for each item or vendor?" much easier to answer than if all items were grouped together in one row per order.

b) In Q5, a missing quantity means that the quantity was not provided or is unknown, while a quantity of zero is a real measurement meaning that the quantity is known and there were 0 units sold for the item. If missing values are treated as zero, calculations such as totals and averages can become inaccurate.